# Packages Loading and config

In [ ]:
import re
import sys
import logging
import structlog
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

import tomli

%load_ext autoreload
%autoreload 2

import pandas as pd
pd.set_option('display.max_rows', 120)
pd.set_option('display.max_columns', 120)

In [ ]:
logging.basicConfig(level=logging.WARNING, stream=sys.stdout)

In [ ]:
import pytanis
from pytanis import GSheetsClient, PretalxClient
from pytanis.pretalx import subs_as_df
from pytanis.review import Col

In [ ]:
# Be aware that this notebook might only run with the following version
pytanis.__version__

In [ ]:
with open('config.toml', 'rb') as fh:
    cfg = tomli.load(fh)

# ⚙️ Configuration

Update the values below before running the notebook:
- `CONF_DATES`: map each abstract day name (as used in the schedule GSheet) to the actual conference date
- `TIMEZONE`: the conference timezone

Slot start times are read directly from the **"Time"** column in the schedule sheet
(format: `"HH:MM - HH:MM"`).  The duration comes from Pretalx.

In [ ]:
TIMEZONE = ZoneInfo('Europe/Berlin')

# Map abstract day names (from the schedule sheet) to actual dates  ← UPDATE THESE
CONF_DATES = {
    'Tuesday': '2026-04-14',
    'Wednesday':  '2026-04-15',
    'Thursday':    '2026-04-16',
}

# Set to False to actually write to Pretalx
DRY_RUN = False

# Read Schedule from Google Sheet

The schedule sheet is expected to be in the format written by `40_scheduling_v1.ipynb`:
- Rows indexed by `(Day, Session, Slot)`
- Columns are room names (possibly with a `Cap: XX%` suffix)
- Cell values are submission codes, either plain or wrapped in `=HYPERLINK(...)` formulas

In [ ]:
gsheet_client = GSheetsClient()  # read-only is sufficient here
worksheet = gsheet_client.gsheet(cfg['schedule_spread_id'], cfg['schedule_work_name'])

# Fetch cell data including hyperlinks (needed because HYPERLINK formulas with newlines
# in the display text are stored as plain text + cell-level hyperlink, not as formula strings)
spreadsheet = worksheet.spreadsheet
sheet_id = worksheet.id
response = spreadsheet.fetch_sheet_metadata(
    params={'includeGridData': True, 'ranges': [worksheet.title]}
)
grid_data = response['sheets'][0]['data'][0]['rowData']

# Extract headers from row 0
header_row = grid_data[0]['values']
headers = [c.get('formattedValue', '') for c in header_row]

def _extract_hyperlink_from_cell(cell: dict) -> str:
    """Return the first hyperlink found in a cell, checking all possible locations."""
    # 1. Cell-level hyperlink (=HYPERLINK formula or Insert > Link)
    if cell.get('hyperlink'):
        return cell['hyperlink']
    # 2. Rich-text hyperlink stored in textFormatRuns
    for run in cell.get('textFormatRuns', []):
        uri = run.get('format', {}).get('link', {}).get('uri', '')
        if uri:
            return uri
    # 3. HYPERLINK formula in userEnteredValue
    formula = cell.get('userEnteredValue', {}).get('formulaValue', '')
    if formula:
        m = re.search(r'HYPERLINK\s*\(\s*"([^"]+)"', formula, re.IGNORECASE)
        if m:
            return m.group(1)
    return ''

# Build per-row data: list of dicts with 'formattedValue' and 'hyperlink' per column
rows_raw = []
for row_data in grid_data[1:]:
    cells = row_data.get('values', [])
    row_cells = []
    for cell in cells:
        row_cells.append({
            'value': cell.get('formattedValue', ''),
            'hyperlink': _extract_hyperlink_from_cell(cell),
        })
    rows_raw.append(row_cells)

print(f"Headers ({len(headers)}): {headers}")
print(f"Data rows: {len(rows_raw)}")

# Debug: show a sample of cells that have hyperlinks
sample = [(h, rc['hyperlink']) for row in rows_raw[:5] for h, rc in zip(headers, row) if rc['hyperlink']]
print(f"Sample hyperlinks found: {sample[:5]}")

In [ ]:
def extract_sub_code_from_hyperlink(hyperlink: str) -> str | None:
    """Extract submission code from a Pretalx submissions URL."""
    if not hyperlink:
        return None
    m = re.search(r'/submissions/([A-Z0-9]+)', hyperlink)
    return m.group(1) if m else None


def clean_room_name(col_name: str) -> str:
    """Remove the 'Cap: XX%' suffix added by 40_scheduling_v1."""
    return re.sub(r'\s*Cap:\s*[\d.]+%\s*$', '', col_name).strip()


def parse_slot_start_time(time_str: str) -> str | None:
    """Parse 'HH:MM - HH:MM' and return the start time as 'HH:MM'."""
    if not time_str:
        return None
    m = re.match(r'(\d{1,2}:\d{2})\s*-\s*\d{1,2}:\d{2}', str(time_str).strip())
    return m.group(1) if m else None


INDEX_COLS = {'Day', 'Session', 'Slot', 'Time'}

records = []
for row_cells in rows_raw:
    row_dict = {headers[i]: row_cells[i] for i in range(min(len(headers), len(row_cells)))}
    day     = row_dict.get('Day', {}).get('value', '').strip()
    session = row_dict.get('Session', {}).get('value', '').strip()
    slot    = row_dict.get('Slot', {}).get('value', '').strip()
    if not day or day == 'nan':
        continue
    time_str   = row_dict.get('Time', {}).get('value', '').strip()
    start_time = parse_slot_start_time(time_str)
    if start_time is None:
        print(f'⚠️  Could not parse Time={time_str!r} for Day={day}, Session={session}, Slot={slot}')
    for col_name, cell in row_dict.items():
        if col_name in INDEX_COLS:
            continue
        sub_code = extract_sub_code_from_hyperlink(cell.get('hyperlink', ''))
        if sub_code:
            records.append({
                'Day': day,
                'Session': session,
                'Slot': slot,
                'StartTime': start_time,
                'Room': clean_room_name(col_name),
                Col.submission: sub_code,
            })

schedule_df = pd.DataFrame(records)
print(f'Loaded {len(schedule_df)} scheduled talks from GSheet')
schedule_df.head(10)

In [ ]:
# Mapping from GSheet room names → Pretalx room names
# ⚠️  Verify 'Hassium' mapping — it has no obvious match in Pretalx
ROOM_MAP = {
    'Spectrum':  'Zeiss Plenary (Spectrum)', #
    'Titanium3': 'Titanium [2nd Floor]', #
    'Helium3':   'Helium [3rd Floor]', # 
    'Europium2': 'Europium [3rd Floor]', #
    'Platinum3':   'Platinum [2nd Floor]', # 
    'Hassium' : 'OpenSpace Lounge [3rd Floor]', # ⚠️ unverified — check this
    'Palladium': 'Palladium [2nd Floor]', #
    'Ferrum':    'Ferrum [Ground Floor]', #
    'Dynamicum': 'Dynamicum [Ground Floor]', #
}

unmapped = set(schedule_df['Room']) - set(ROOM_MAP)
if unmapped:
    print(f'⚠️  Rooms in GSheet with no mapping: {unmapped}')
else:
    print('✅ All GSheet rooms have a mapping')

schedule_df['Room'] = schedule_df['Room'].map(ROOM_MAP).fillna(schedule_df['Room'])
print(schedule_df['Room'].unique())

# Load Data from Pretalx

In [ ]:
pretalx_client = PretalxClient(blocking=True)

In [ ]:
# Rooms: name → id
rooms_count, rooms_iter = pretalx_client.rooms(cfg['event_name'])
rooms_list = list(rooms_iter)
room_name_to_id = {room.name.en: room.id for room in rooms_list}
print('Rooms found in Pretalx:')
for name, rid in room_name_to_id.items():
    print(f'  {name!r} → id={rid}')

In [ ]:
# WIP schedule slots: submission code → slot id
slots_count, slots_iter = pretalx_client.wip_slots(cfg['event_name'])
slots_list = list(slots_iter)
print(f'Found {len(slots_list)} WIP slots')

sub_to_slot_id: dict[str, int] = {}
for s in slots_list:
    sub = s.get('submission')
    if sub:
        sub_to_slot_id[sub] = s['id']

print(f'Slots linked to a submission: {len(sub_to_slot_id)}')

In [ ]:
# Confirmed/accepted talks: submission code → duration (minutes)
talks_count, talks_iter = pretalx_client.submissions(
    cfg['event_name'], params={'state': ['confirmed', 'accepted']}
)
talks_list = list(talks_iter)

# Duration source 1: WIP slots already carry the resolved duration (submission or type default)
talk_duration: dict[str, int] = {
    s['submission']: s['duration']
    for s in slots_list
    if s.get('submission') and s.get('duration')
}

# Duration source 2: submission's own field → fall back to submission type default
if len(talk_duration) < len(talks_list):
    sub_types_count, sub_types_iter = pretalx_client.submission_types(cfg['event_name'])
    sub_type_default_duration = {st.id: st.default_duration for st in sub_types_iter}
    for t in talks_list:
        if t.code in talk_duration:
            continue
        dur = t.duration
        if dur is None and t.submission_type_id is not None:
            dur = sub_type_default_duration.get(t.submission_type_id)
        if dur:
            talk_duration[t.code] = dur

print(f'Loaded durations for {len(talk_duration)} talks')

# Build Schedule Updates

Each row in the schedule sheet carries a **"Time"** column (e.g. `"11:45 - 12:10"`)
that gives the start time for that `(Day, Session, Slot)` block.
Talks in the same slot run in parallel across rooms, so every room gets the same start
time for a given slot.

In [ ]:
def get_slot_start(day: str, start_time: str) -> datetime:
    """Return the timezone-aware start datetime for a slot."""
    date_str = CONF_DATES[day]
    return datetime.fromisoformat(f'{date_str}T{start_time}:00').replace(tzinfo=TIMEZONE)


update_rows = []

for _, row in schedule_df.iterrows():
    day      = row['Day']
    session  = row['Session']
    slot     = row['Slot']
    room     = row['Room']
    sub_code = row[Col.submission]
    start_time = row['StartTime']

    if start_time is None:
        print(f'⚠️  No start time for {sub_code} ({day}/{session}/{slot}) – skipping')
        continue

    duration_min = talk_duration.get(sub_code)
    if duration_min is None:
        print(f'⚠️  No duration found for {sub_code} – skipping')
        continue

    current_start = get_slot_start(day, start_time)
    talk_end = current_start + timedelta(minutes=duration_min)
    update_rows.append({
        'Day': day,
        'Session': session,
        'Slot': slot,
        'Room': room,
        Col.submission: sub_code,
        'room_id': room_name_to_id.get(room),
        'slot_id': sub_to_slot_id.get(sub_code),
        'start': current_start.isoformat(),
        'end': talk_end.isoformat(),
    })

updates_df = pd.DataFrame(update_rows)
print(f'Built {len(updates_df)} schedule updates')
updates_df

# Validate

Check that every row has a valid room_id and slot_id before writing.

In [ ]:
missing_room = updates_df[updates_df['room_id'].isna()]
if not missing_room.empty:
    print('⚠️  Rooms not found in Pretalx (check spelling):')
    print(missing_room[['Room', Col.submission]].to_string(index=False))
else:
    print('✅ All rooms matched')

missing_slot = updates_df[updates_df['slot_id'].isna()]
if not missing_slot.empty:
    print('\n⚠️  No WIP slot found for these submissions (not yet confirmed?):')
    print(missing_slot[[Col.submission, 'Room']].to_string(index=False))
else:
    print('✅ All submissions have a WIP slot')

sched_codes = set(updates_df[Col.submission])
gsheet_codes = set(schedule_df[Col.submission])
dropped = gsheet_codes - sched_codes
if dropped:
    print(f'\n⚠️  {len(dropped)} talk(s) from GSheet were skipped (missing duration or slot): {dropped}')

In [ ]:
# Check for time overlaps within the same room on the same day
overlaps = []
for (day, room), grp in updates_df.groupby(['Day', 'Room']):
    grp = grp.sort_values('start')
    ends = pd.to_datetime(grp['end'].values[:-1])
    starts = pd.to_datetime(grp['start'].values[1:])
    if (ends > starts).any():
        overlaps.append((day, room))

if overlaps:
    print('⚠️  Time overlaps detected in:')
    for pair in overlaps:
        print(f'  Day={pair[0]}, Room={pair[1]}')
else:
    print('✅ No time overlaps detected')

# Write Schedule to Pretalx

Set `DRY_RUN = False` in the configuration cell to actually write.

The notebook PATCHes each WIP slot with:
- `room`: the Pretalx room ID
- `start`: ISO 8601 datetime (timezone-aware)

`end` is **not** sent — Pretalx computes it automatically from `start + submission duration`.

Only slots in the current WIP schedule can be changed.  Frozen schedule versions are read-only.

In [ ]:
valid_updates = updates_df.dropna(subset=['room_id', 'slot_id'])
print(f'Will update {len(valid_updates)} / {len(updates_df)} slots  (DRY_RUN={DRY_RUN})')

errors = []

for i, (_, row) in enumerate(valid_updates.iterrows()):
    slot_id  = int(row['slot_id'])
    room_id  = int(row['room_id'])
    sub_code = row[Col.submission]
    start    = row['start']

    if DRY_RUN:
        print(f'[DRY RUN] PATCH slot {slot_id:4d}  sub={sub_code}  room_id={room_id}  start={start}')
    else:
        try:
            # Note: do NOT send 'end' — Pretalx derives it from start + submission duration
            result = pretalx_client.patch_slot(
                cfg['event_name'],
                slot_id,
                {'room': room_id, 'start': start},
            )
            msg = f'✅ slot {slot_id:4d}  sub={sub_code}'
            # Print full result for first 3 updates so we can verify
            if i < 3:
                msg += f'  → room={result.get("room")}  start={result.get("start")}  end={result.get("end")}'
            print(msg)
        except Exception as exc:
            body = getattr(getattr(exc, 'response', None), 'text', '')
            print(f'❌ slot {slot_id:4d}  sub={sub_code}  ERROR: {exc}')
            if body:
                print(f'   response body: {body}')
            errors.append({'slot_id': slot_id, Col.submission: sub_code, 'error': str(exc), 'body': body})
            break

if errors:
    print(f'\n{len(errors)} errors occurred.')
elif not DRY_RUN:
    print(f'\n✅ All {len(valid_updates)} slots written successfully')

# (Optional) Release Schedule

After verifying the WIP schedule looks correct in the Pretalx UI you can freeze it
as a named version.  This publishes the schedule to the conference website.

```python
import httpx, pytanis
from pytanis import get_cfg

cfg_pytanis = get_cfg()
headers = {
    'Authorization': f'Token {cfg_pytanis.Pretalx.api_token}',
    'Pretalx-Version': cfg_pytanis.Pretalx.api_version,
    'Content-Type': 'application/json',
}
resp = httpx.post(
    f'{cfg_pytanis.Pretalx.api_base_url}/api/events/{cfg["event_name"]}/schedules/release/',
    json={'version': '1.0', 'comment': 'Initial schedule'},
    headers=headers,
)
resp.raise_for_status()
print(resp.json())
```